# Issue173: live environment observation only
Private notebook; attach no Inputs. No dataset/checkpoint reads, weight load or inference.
Side effects: source/receipt/ZIP files, two simultaneous 1 MiB scratch/output files (removed), a tiny FP16 tensor probe on each visible GPU. Repeat creates a new observation.
GPU quota/time is used; no production attempt or model call is consumed. Kernel CUDA is not pinned-worker or model-fit verification. Export all ZIPs and sidecars.


In [ ]:
import base64, hashlib, uuid
from pathlib import Path
payload = base64.b64decode('IiIiSXNzdWUxNzMgbm8tSW5wdXQgb2JzZXJ2YXRpb24uIE5vIHdlaWdodHMsIGRhdGFzZXQgcmVhZGVyIG9yIGluZmVyZW5jZSBlbnRyeS4KCkV4cGxpY2l0IGludm9jYXRpb24gb25seS4gUmVwZWF0IHJ1bnMgZ2V0IHVuaXF1ZSBvdXRwdXQ7IGZhaWxlZCBjaGVja3Mgc3Vydml2ZS4KVGhlIGtlcm5lbCB0ZW5zb3IgcHJvYmUgZG9lcyBub3QgcXVhbGlmeSBhbnkgcGlubmVkIHByb2R1Y3Rpb24gd29ya2VyLgoiIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHV1aWQKaW1wb3J0IHppcGZpbGUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCgoKZGVmIHdyaXRlKHBhdGgsIHZhbHVlKToKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCd4JywgZW5jb2Rpbmc9J3V0Zi04JykgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh2YWx1ZSwgc3RyZWFtLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUsIGFsbG93X25hbj1GYWxzZSkKICAgICAgICBzdHJlYW0ud3JpdGUoJ1xuJykKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgc2hhKHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDIqKjIwKSwgYicnKToKICAgICAgICAgICAgaC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBhbGxvY2F0aW9uKHNjcmF0Y2gsIG91dHB1dCwgc2l6ZT0yKioyMCk6CiAgICBpZiB0eXBlKHNpemUpIGlzIG5vdCBpbnQgb3Igbm90IDAgPCBzaXplIDw9IDE2ICogMioqMjA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignT0JTRVJWQVRJT05fQUxMT0NBVElPTl9CT1VORCcpCiAgICBjcmVhdGVkLCByb3dzID0gW10sIFtdCiAgICB0cnk6CiAgICAgICAgZm9yIHJvb3QgaW4gKFBhdGgoc2NyYXRjaCksIFBhdGgob3V0cHV0KSk6CiAgICAgICAgICAgIHBhdGggPSByb290IC8gKCdwcm9iZS0nICsgdXVpZC51dWlkNCgpLmhleCkKICAgICAgICAgICAgd2l0aCBwYXRoLm9wZW4oJ3hiJykgYXMgc3RyZWFtOgogICAgICAgICAgICAgICAgY3JlYXRlZC5hcHBlbmQocGF0aCkKICAgICAgICAgICAgICAgIHN0cmVhbS53cml0ZShiJ1x4YTUnICogc2l6ZSkKICAgICAgICAgICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgICAgICAgICBvcy5mc3luYyhzdHJlYW0uZmlsZW5vKCkpCiAgICAgICAgICAgIHN0ID0gcGF0aC5zdGF0KCkKICAgICAgICAgICAgYWxsb2NhdGVkID0gc3Quc3RfYmxvY2tzICogNTEyIGlmIGhhc2F0dHIoc3QsICdzdF9ibG9ja3MnKSBlbHNlIE5vbmUKICAgICAgICAgICAgaWYgc3Quc3Rfc2l6ZSAhPSBzaXplIG9yIChhbGxvY2F0ZWQgaXMgbm90IE5vbmUgYW5kIGFsbG9jYXRlZCA8IHNpemUpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUEhZU0lDQUxfQUxMT0NBVElPTl9OT1RfUFJPVkVOJykKICAgICAgICAgICAgcm93cy5hcHBlbmQoeyd3cml0dGVuX2J5dGVzJzogc3Quc3Rfc2l6ZSwgJ2FsbG9jYXRlZF9ieXRlcyc6IGFsbG9jYXRlZCwKICAgICAgICAgICAgICAgICAgICAgICAgICdkZXZpY2UnOiBzdC5zdF9kZXYsICdmcmVlX2J5dGVzJzogc2h1dGlsLmRpc2tfdXNhZ2Uocm9vdCkuZnJlZX0pCiAgICBmaW5hbGx5OgogICAgICAgIGZvciBwYXRoIGluIGNyZWF0ZWQ6CiAgICAgICAgICAgIHBhdGgudW5saW5rKCkKICAgIHJldHVybiB7J2FsbG9jYXRpb25zJzogcm93cywgJ2NsZWFudXBfdmVyaWZpZWQnOiBhbGwobm90IHAuZXhpc3RzKCkgZm9yIHAgaW4gY3JlYXRlZCksCiAgICAgICAgICAgICdzYW1lX2RldmljZSc6IHJvd3NbMF1bJ2RldmljZSddID09IHJvd3NbMV1bJ2RldmljZSddLAogICAgICAgICAgICAncXVvdGFfc2NvcGUnOiAnU01BTExfUFJPQkVfT05MWV9OT1RfRlVMTF9NT0RFTF9DQVBBQ0lUWSd9CgoKZGVmIGNvbW1hbmQoYXJndiwgdGltZW91dD00NSk6CiAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bihhcmd2LCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIHRpbWVvdXQ9dGltZW91dCkKICAgIGlmIGxlbihyZXN1bHQuc3Rkb3V0KSArIGxlbihyZXN1bHQuc3RkZXJyKSA+IDY1NTM2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09CU0VSVkFUSU9OX09VVFBVVF9CT1VORCcpCiAgICByZXR1cm4geydyZXR1cm5jb2RlJzogcmVzdWx0LnJldHVybmNvZGUsICdzdGRvdXQnOiByZXN1bHQuc3Rkb3V0LCAnc3RkZXJyJzogcmVzdWx0LnN0ZGVycn0KCgpkZWYgbWVtb3J5KCk6CiAgICBmaWVsZHMgPSBkaWN0KGxpbmUuc3BsaXQoJzonLCAxKSBmb3IgbGluZSBpbiBQYXRoKCcvcHJvYy9tZW1pbmZvJykucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpKQogICAgcmVzdWx0ID0geydob3N0X2F2YWlsYWJsZV9ieXRlcyc6IGludChmaWVsZHNbJ01lbUF2YWlsYWJsZSddLnNwbGl0KClbMF0pICogMTAyNCwKICAgICAgICAgICAgICAnY2dyb3VwJzoge319CiAgICBmb3IgcmVsIGluICgnbWVtb3J5Lm1heCcsICdtZW1vcnkuY3VycmVudCcsICdtZW1vcnkvbWVtb3J5LmxpbWl0X2luX2J5dGVzJywKICAgICAgICAgICAgICAgICdtZW1vcnkvbWVtb3J5LnVzYWdlX2luX2J5dGVzJyk6CiAgICAgICAgcCA9IFBhdGgoJy9zeXMvZnMvY2dyb3VwJykgLyByZWwKICAgICAgICBpZiBwLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmVzdWx0WydjZ3JvdXAnXVtyZWxdID0gcC5yZWFkX3RleHQoKS5zdHJpcCgpCiAgICByZXR1cm4gcmVzdWx0CgoKQ1VEQV9QUk9CRSA9ICcnJwppbXBvcnQganNvbiwgdG9yY2gKcmVzdWx0ID0geyd0b3JjaCc6IHRvcmNoLl9fdmVyc2lvbl9fLCAnY3VkYV9idWlsZCc6IHRvcmNoLnZlcnNpb24uY3VkYSwKICAgICAgICAgICdhdmFpbGFibGUnOiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpLCAnZGV2aWNlcyc6IFtdfQpmb3IgaSBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKToKICAgIHggPSB0b3JjaC5vbmVzKCgyLCAyKSwgZGV2aWNlPWYnY3VkYTp7aX0nLCBkdHlwZT10b3JjaC5mbG9hdDE2KQogICAgeSA9IHggQCB4CiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGkpCiAgICByZXN1bHRbJ2RldmljZXMnXS5hcHBlbmQoeydpbmRleCc6IGksICduYW1lJzogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoaSksCiAgICAgICAgJ2NhcGFiaWxpdHknOiBsaXN0KHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KGkpKSwKICAgICAgICAnbWVtb3J5X2J5dGVzJzogbGlzdCh0b3JjaC5jdWRhLm1lbV9nZXRfaW5mbyhpKSksICdmcDE2X21hdG11bCc6IHkudG9saXN0KCl9KQpwcmludChqc29uLmR1bXBzKHJlc3VsdCkpCicnJwoKCmRlZiBvYnNlcnZlKHBhcmVudCwgc2NyYXRjaCwgKiwgc291cmNlX2NvbW1pdCwgZ3B1PVRydWUpOgogICAgaWYgbGVuKHNvdXJjZV9jb21taXQpICE9IDQwIG9yIGFueShjIG5vdCBpbiAnMDEyMzQ1Njc4OWFiY2RlZicgZm9yIGMgaW4gc291cmNlX2NvbW1pdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRVhBQ1RfU09VUkNFX0NPTU1JVF9SRVFVSVJFRCcpCiAgICBwYXJlbnQsIHNjcmF0Y2ggPSBQYXRoKHBhcmVudCksIFBhdGgoc2NyYXRjaCkKICAgIGlmIGFueShwLmlzX3N5bWxpbmsoKSBmb3Igcm9vdCBpbiAocGFyZW50LCBzY3JhdGNoKSBmb3IgcCBpbiAocm9vdCwgKnJvb3QucGFyZW50cykpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ09VVFBVVF9BTElBUycpCiAgICBwYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc2NyYXRjaC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBydW5faWQgPSAnaXNzdWUxNzMtZW52LScgKyB1dWlkLnV1aWQ0KCkuaGV4CiAgICByb290ID0gcGFyZW50IC8gcnVuX2lkCiAgICByb290Lm1rZGlyKCkKICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgcmVwb3J0ID0geydzY2hlbWEnOiAnc2FmZXNoaWZ0LWlzc3VlMTczLW9ic2VydmF0aW9uLXYxJywgJ3J1bl9pZCc6IHJ1bl9pZCwKICAgICAgICAnc291cmNlX2NvbW1pdCc6IHNvdXJjZV9jb21taXQsICdzY3JpcHRfc2hhMjU2Jzogc2hhKF9fZmlsZV9fKSwKICAgICAgICAndXRjJzogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksICdtb2RlbF9jYWxscyc6IDAsCiAgICAgICAgJ3dlaWdodHNfbG9hZGVkJzogRmFsc2UsICdkYXRhc2V0X2FjY2Vzc2VkJzogRmFsc2UsICdvd25lcl9zbG90X2NvbnN1bWVkJzogRmFsc2UsCiAgICAgICAgJ3NjaWVudGlmaWNfcXVhbGlmaWVkJzogRmFsc2UsICdwcm9kdWN0aW9uX2F1dGhvcml6ZWQnOiBGYWxzZSwKICAgICAgICAnZGF0YV9nYXRlJzogJ0RBVEFfUklHSFRTX0FORF9GUkVTSF9DVVNUT0RZX05PVF9WRVJJRklFRCcsCiAgICAgICAgJ2NoZWNrcG9pbnRfZ2F0ZSc6ICdOT1RfQVRUQUNIRURfTk9UX1ZFUklGSUVEJywKICAgICAgICAnd29ya2VyX2dhdGUnOiAnUElOTkVEX1dPUktFUlNfTk9UX1ZFUklGSUVEJywgJ2NoZWNrcyc6IHt9fQogICAgZGVmIGNoZWNrKG5hbWUsIGZuKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZhbHVlID0gZm4oKQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KSBhbmQgdmFsdWUuZ2V0KCdyZXR1cm5jb2RlJywgMCkgIT0gMDoKICAgICAgICAgICAgICAgIHJlcG9ydFsnY2hlY2tzJ11bbmFtZV0gPSB7J3N0YXR1cyc6ICdGQUlMJywgJ3ZhbHVlJzogdmFsdWV9CiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICByZXBvcnRbJ2NoZWNrcyddW25hbWVdID0geydzdGF0dXMnOiAnT0JTRVJWRUQnLCAndmFsdWUnOiB2YWx1ZX0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgcmVwb3J0WydjaGVja3MnXVtuYW1lXSA9IHsnc3RhdHVzJzogJ0ZBSUwnLCAnZXJyb3JfdHlwZSc6IHR5cGUoZXhjKS5fX25hbWVfXywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdlcnJvcic6IHN0cihleGMpWzoyMDAwXX0KICAgIGNoZWNrKCdrZXJuZWwnLCBsYW1iZGE6IHsncHl0aG9uJzogcGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwgJ2ltcGxlbWVudGF0aW9uJzogcGxhdGZvcm0ucHl0aG9uX2ltcGxlbWVudGF0aW9uKCksCiAgICAgICAgJ3BsYXRmb3JtJzogcGxhdGZvcm0ucGxhdGZvcm0oKSwgJ2FyY2hpdGVjdHVyZSc6IHBsYXRmb3JtLm1hY2hpbmUoKSwKICAgICAgICAnYWJpJzogc3lzLmltcGxlbWVudGF0aW9uLmNhY2hlX3RhZ30pCiAgICBjaGVjaygncGFja2FnZXMnLCBsYW1iZGE6IHtuOiBtZXRhZGF0YS52ZXJzaW9uKG4pIGZvciBuIGluCiAgICAgICAgKCdwaXAnLCAndG9yY2gnLCAndG9yY2h2aXNpb24nLCAndHJhbnNmb3JtZXJzJywgJ251bXB5JywgJ3BpbGxvdycpfSkKICAgIGNoZWNrKCdtZW1vcnknLCBtZW1vcnkpCiAgICBjaGVjaygnc3RvcmFnZScsIGxhbWJkYTogYWxsb2NhdGlvbihzY3JhdGNoLCByb290KSkKICAgIGlmIGdwdToKICAgICAgICBjaGVjaygnZHJpdmVyJywgbGFtYmRhOiBjb21tYW5kKFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCxuYW1lLGRyaXZlcl92ZXJzaW9uLG1lbW9yeS50b3RhbCxtZW1vcnkuZnJlZScsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJy0tZm9ybWF0PWNzdixub2hlYWRlcixub3VuaXRzJ10sIDEwKSkKICAgICAgICBjaGVjaygna2VybmVsX2N1ZGFfdGVuc29yX29ubHknLCBsYW1iZGE6IGNvbW1hbmQoW3N5cy5leGVjdXRhYmxlLCAnLUknLCAnLUInLCAnLWMnLCBDVURBX1BST0JFXSkpCiAgICBlbHNlOgogICAgICAgIHJlcG9ydFsnY2hlY2tzJ11bJ2tlcm5lbF9jdWRhX3RlbnNvcl9vbmx5J10gPSB7J3N0YXR1cyc6ICdOT1RfUlVOJ30KICAgIHJlcG9ydFsnc2Vjb25kcyddID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0CiAgICByZXBvcnRbJ3N0YXR1cyddID0gJ0VOVklST05NRU5UX09CU0VSVkVEJyBpZiBhbGwodlsnc3RhdHVzJ10gIT0gJ0ZBSUwnIGZvciB2IGluIHJlcG9ydFsnY2hlY2tzJ10udmFsdWVzKCkpIGVsc2UgJ09CU0VSVkVEX1dJVEhfRkFJTFVSRVMnCiAgICByZXBvcnRbJ2luZmVyZW5jZV9zdGF0dXMnXSA9ICdOT1RfQVRURU1QVEVEJwogICAgd3JpdGUocm9vdCAvICdvYnNlcnZhdGlvbi5qc29uJywgcmVwb3J0KQogICAgd3JpdGUocm9vdCAvICdjaGVja3N1bXMuanNvbicsIHsnb2JzZXJ2YXRpb24uanNvbic6IHNoYShyb290IC8gJ29ic2VydmF0aW9uLmpzb24nKX0pCiAgICBhcmNoaXZlID0gcGFyZW50IC8gKHJ1bl9pZCArICcuemlwJykKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGFyY2hpdmUsICd4JywgY29tcHJlc3Npb249emlwZmlsZS5aSVBfREVGTEFURUQpIGFzIGJ1bmRsZToKICAgICAgICBmb3IgcGF0aCBpbiBzb3J0ZWQocm9vdC5pdGVyZGlyKCkpOgogICAgICAgICAgICBidW5kbGUud3JpdGUocGF0aCwgcGF0aC5uYW1lKQogICAgd3JpdGUoUGF0aChzdHIoYXJjaGl2ZSkgKyAnLnNoYTI1Ni5qc29uJyksIHsnc2hhMjU2Jzogc2hhKGFyY2hpdmUpLCAnc2l6ZSc6IGFyY2hpdmUuc3RhdCgpLnN0X3NpemUsCiAgICAgICAgJ21vZGVsX2NhbGxzJzogMCwgJ3NvdXJjZV9jb21taXQnOiBzb3VyY2VfY29tbWl0fSkKICAgIHJldHVybiByb290CgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIGltcG9ydCBhcmdwYXJzZQogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHAuYWRkX2FyZ3VtZW50KCctLW91dHB1dCcsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1zY3JhdGNoJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCctLXNvdXJjZS1jb21taXQnLCByZXF1aXJlZD1UcnVlKQogICAgYSA9IHAucGFyc2VfYXJncygpCiAgICBwcmludChvYnNlcnZlKGEub3V0cHV0LCBhLnNjcmF0Y2gsIHNvdXJjZV9jb21taXQ9YS5zb3VyY2VfY29tbWl0KSkK')
assert hashlib.sha256(payload).hexdigest() == '54c8759a5a9899d90c616bbebeb1947e48744732f1230e42e3c4a6e163f7fb39'
root = Path('/kaggle/working') / ('issue173-source-' + uuid.uuid4().hex)
root.mkdir(exist_ok=False)
script = root / 'observe_kaggle173.py'
with script.open('xb') as stream:
    stream.write(payload)
namespace = {'__file__': str(script), '__name__': 'issue173_observation'}
exec(compile(payload, str(script), 'exec'), namespace)
receipt = namespace['observe']('/kaggle/working', '/kaggle/temp/issue173', source_commit='dd152630877c29faf9f4439d902e55720e823378')
print(receipt)
